# Ranked list preparation

- Take gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy
- Make rank list of genes based on TA pleiotropy count
- Apply z-score correction

In [4]:
from pathlib import Path

from gentropy.common.session import Session

gcs_genes_pleiotropy = "gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy"
filepath = str(Path("../../../data/genes_pleiotropy").resolve())

session = Session(extended_spark_conf={"spark.driver.memory": "10g"})


In [ ]:
# Download genes_pleiotropy parquet from GCS to data folder (run once; requires gcloud auth)
import subprocess

local_dir = Path("../../../data/genes_pleiotropy").resolve()
local_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        "gcloud",
        "storage",
        "--billing-project=open-targets-genetics-dev",
        "rsync",
        "-r",
        gcs_genes_pleiotropy,
        str(local_dir),
    ],
    check=True,
)


At gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy/**, worker process 42776 thread 8625679424 listed 12...
Copying gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy/._SUCCESS.crc to file:///Users/polina/Gentropy-manuscript/data/genes_pleiotropy/._SUCCESS.crc
Copying gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy/.part-00000-bc3ff23c-243e-4360-a61a-61cb7c42af54-c000.snappy.parquet.crc to file:///Users/polina/Gentropy-manuscript/data/genes_pleiotropy/.part-00000-bc3ff23c-243e-4360-a61a-61cb7c42af54-c000.snappy.parquet.crc
  
Copying gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy/.part-00001-bc3ff23c-243e-4360-a61a-61cb7c42af54-c000.snappy.parquet.crc to file:///Users/polina/Gentropy-manuscript/data/genes_pleiotropy/.part-00001-bc3ff23c-243e-4360-a61a-61cb7c42af54-c000.snappy.parquet.crc
Copying gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy/.part-00002-bc3ff23c-243e-4360-a61a-61cb7c42af54-c000.snappy.parquet.crc 

CompletedProcess(args=['gcloud', 'storage', '--billing-project=open-targets-genetics-dev', 'rsync', '-r', 'gs://genetics-portal-dev-analysis/dc16/output/genes_pleiotropy', '/Users/polina/Gentropy-manuscript/data/genes_pleiotropy'], returncode=0)

In [6]:
def read_genes_pleiotropy_parquet(spark_session, path=None):
    """Read genes_pleiotropy parquet using PySpark and return the DataFrame. Uses local data folder by default."""
    if path is None:
        path = filepath
    return spark_session.read.parquet(path)


# Read from local data folder and show headers (column names)
genes_pleiotropy_df = read_genes_pleiotropy_parquet(session.spark)
print("Headers (columns):", genes_pleiotropy_df.columns)
genes_pleiotropy_df.printSchema()


Headers (columns): ['geneId', 'uniqueVariants', 'uniqueDiseases', 'uniqueTherapeuticAreas', 'maxEQTLColoc', 'maxPQTLColoc', 'maxVEP', 'maxDistanceTSS', 'minEffectiveSampleSize', 'maxEffectiveSampleSize', 'earliestPublicationDate', 'cancerOrBenignTumor', 'infectiousDisease', 'pregnancyOrPerinatalDisease', 'disorderOfVisualSystem', 'cardiovascularDisease', 'pancreasDisease', 'gastrointestinalDisease', 'reproductiveSystemOrBreastDisease', 'integumentarySystemDisease', 'endocrineSystemDisease', 'respiratoryOrThoracicDisease', 'urinarySystemDisease', 'musculoskeletalOrConnectiveTissueDisease', 'disorderOfEar', 'immuneSystemDisease', 'hematologicDisease', 'nervousSystemDisease', 'psychiatricDisorder', 'nutritionalOrMetabolicDisease', 'geneticFamilialOrCongenitalDisease', 'injuryPoisoningOrOtherComplication', 'signOrSymptom', 'other', 'totalStudies', 'approvedSymbol', 'lofConstraint', 'misConstraint', 'synConstraint', 'pathwayCount', 'geneLength', 'tissueSpecificity', 'tissueDistribution', 'n

In [8]:
import pyspark.sql.functions as F

# Z-score: (x - mean) / std over uniqueTherapeuticAreas
stats = genes_pleiotropy_df.agg(
    F.mean("uniqueTherapeuticAreas").alias("mean"),
    F.stddev("uniqueTherapeuticAreas").alias("std"),
).collect()[0]
mean_ta, std_ta = stats["mean"], stats["std"]
if std_ta is None or std_ta == 0:
    std_ta = 1.0

ranked = (
    genes_pleiotropy_df.withColumn(
        "globalScore",
        (F.col("uniqueTherapeuticAreas") - mean_ta) / std_ta,
    )
    .select(
        "globalScore",
        F.col("geneId").alias("symbol"),
    )
    .orderBy(F.desc("globalScore"))
)

out_path = Path("../../../data/for_gsea/geneset_ta_pleiotropy_zscore.tsv").resolve()
out_path.parent.mkdir(parents=True, exist_ok=True)
ranked.toPandas().to_csv(out_path, sep="\t", index=False)
print(f"Written: {out_path}")


Written: /Users/polina/Gentropy-manuscript/data/for_gsea/geneset_ta_pleiotropy_zscore.tsv


# GSEA

GSEA is performed using blitzgsea with pathway library gene sets (Reactome and KEGG) as background.

In [2]:
import os
from pathlib import Path
import pandas as pd
import blitzgsea as blitz


In [ ]:
def load_custom_gmt(path):
    """
    Parse a GMT file into a dict: {term_name: [gene1, gene2, ...], …}
    """
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"GMT file not found: {path}")

    with path.open("r") as f:
        return {
            parts[0]: parts[2:]  # skip description at index 1
            for line in f
            if (parts := line.strip().split("	")) and len(parts) > 2
        }


def run_gsea_pandas(
    input_tsv,
    gmt_file,
    output_tsv=None,
    processes=4,
    symbol_col="symbol",
    score_col="globalScore",
):
    """
    Reads TSV, renames columns, runs GSEA with custom pathways, saves as TSV.

    Parameters
    ----------
    input_tsv : str
        Input file with at least the columns specified by symbol_col and score_col.
    gmt_file : str
        Path to custom GMT file.
    processes : int
        Number of processes for GSEA.
    output_tsv : str or None
        Custom output filename. If None, defaults to <input_basename>_gsea.tsv.
    symbol_col : str
        Column name containing gene symbols (default "symbol").
    score_col : str
        Column name containing scores (default "globalScore").
    """
    input_path = Path(input_tsv)
    if not input_path.is_file():
        raise FileNotFoundError(f"Input TSV not found: {input_path}")

    gmt_path = Path(gmt_file)
    if not gmt_path.is_file():
        raise FileNotFoundError(f"GMT file not found: {gmt_path}")

    library_sets = load_custom_gmt(gmt_path)
    if not library_sets:
        raise ValueError(f"No pathways found in {gmt_path}")

    df = pd.read_csv(input_path, sep="	", header=0, index_col=None)

    missing_cols = {symbol_col, score_col} - set(df.columns)
    if missing_cols:
        raise ValueError(
            f"Missing required columns in input TSV: {sorted(missing_cols)}"
        )

    gsea_df = pd.DataFrame()
    gsea_df[1] = df[symbol_col]
    gsea_df[0] = pd.to_numeric(df[score_col], errors="coerce")
    gsea_df = gsea_df.dropna(subset=[0])

    print(f"GSEA input shape: {gsea_df.shape}")
    print(f"GSEA input columns: {gsea_df.columns.tolist()}")
    print("GSEA input sample:")
    print(gsea_df.head())

    res_df = blitz.gsea(gsea_df, library_sets, processes=processes).reset_index(
        names="Term"
    )

    res_df["propagated_edge"] = res_df["Term"].apply(
        lambda t: ",".join(library_sets.get(t, [])) if library_sets.get(t) else ""
    )

    term_series = res_df["Term"]
    res_df["Source"] = term_series.str.extract(r"\{([^}]+)\}", expand=False).fillna("")
    res_df["Term"] = term_series.str.replace(r"\{[^}]+\}", "", regex=True)
    res_df["Term"] = (
        res_df["Term"].str.replace(r"\s*\[[^\]]+\]", "", regex=True).str.strip()
    )

    if "leading_edge" in res_df.columns:
        res_df["leading_edge"] = res_df["leading_edge"].apply(
            lambda x: ",".join(map(str, x)) if isinstance(x, (list, tuple)) else str(x)
        )

    first_cols = ["Term", "Source"]
    res_df = res_df[first_cols + [c for c in res_df.columns if c not in first_cols]]

    output_path = (
        Path(output_tsv)
        if output_tsv is not None
        else input_path.with_name(f"{input_path.stem}_gsea.tsv")
    )

    res_df.to_csv(output_path, sep="	", index=False)
    print(f"GSEA results saved to {output_path}")
    return res_df


In [9]:
scores = "..//..//..//data/for_gsea/geneset_ta_zscore.tsv"
library = "..//..//..//data/gene_sets/gene_sets.gmt"
output_name = "..//..//..//data/gsea_results/geneset_ta_gsea.tsv"

run_gsea_pandas(scores, library, processes=4, output_tsv=output_name)


GSEA input shape: (8285, 2)
GSEA input columns: [1, 0]
GSEA input sample:
        1         0
0  CDKN2B  8.694865
1     ABO  8.224087
2     FTO  7.753310
3   SH2B3  6.811755
4    APOE  6.340978
GSEA results saved to ../../../data/gsea_results/geneset_ta_gsea.tsv


,Term,Source,es,nes,pval,sidak,fdr,geneset_size,leading_edge,propagated_edge
0,Regulation of Transcription by RNA Polymerase ...,GO_Biological_Process_2025,0.469119,8.264785,1.399467e-16,8.339423e-13,8.339423e-13,1136,"ZNF827,SMAD3,CAMK2D,NOD2,TCF7L2,VEGFA,IRF1,PPA...","HEXIM2,HEXIM1,ESX1,PASD1,ISL1,ISL2,LARP7,TFR2,..."
1,Regulation of DNA-templated Transcription (GO:...,GO_Biological_Process_2025,0.461036,7.618945,2.557567e-14,1.524054e-10,7.620270e-11,1058,"CELSR2,MAGI1,ZNF827,APOE,SMAD3,GDF7,VGLL4,SLC3...","HEXIM2,ESX1,PASD1,IL31RA,NOBOX,SKAP1,PTF1A,ATF..."
2,Positive Regulation of DNA-templated Transcrip...,GO_Biological_Process_2025,0.509503,7.445169,9.682069e-14,5.769545e-10,1.923182e-10,770,"GDF7,ZNF827,APOE,SMAD3,NOD2,TCF7L2,VEGFA,IRF1,...","ZNF281,DEK,GATAD2A,RFX3,MRTFB,RFX4,MRTFA,ISL1,..."
3,Negative Regulation of DNA-templated Transcrip...,GO_Biological_Process_2025,0.539275,7.051514,1.769818e-12,1.054635e-08,2.636587e-09,562,"MAGI1,ZNF827,VGLL4,SMAD3,TCF7L2,VEGFA,IRF1,PPA...","HEXIM2,RBFOX2,HEXIM1,ESX1,ZNF282,ZNF281,DICER1..."
4,Positive Regulation of Transcription by RNA Po...,GO_Biological_Process_2025,0.520016,6.839356,7.955016e-12,4.740394e-08,9.480788e-09,604,"ZNF827,SMAD3,NOD2,TCF7L2,VEGFA,IRF1,PPARG,GLIS...","DEK,RFX3,MRTFB,RFX4,MRTFA,ISL1,ISL2,LARP7,RFX7..."
...,...,...,...,...,...,...,...,...,...,...
5954,Supramolecular Fiber Organization (GO:0097435),GO_Biological_Process_2025,0.281790,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,155,"MYO1H,TGFB2,COL11A1,SLK,ELMO1,CDC42,ACTN1,MAPT...","CARMIL1,KRT1,LUM,KRT5,KRT4,KRT3,ACTN1,KRT2,KRT..."
5955,Sulfur Compound Transport (GO:0072348),GO_Biological_Process_2025,0.235036,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,12,"SLC26A5,SLC22A1,SLC19A2,MGST1","NHERF1,SLC13A4,SLC13A1,SLC26A9,SLC26A8,SLC25A1..."
5956,Steroid Catabolic Process (GO:0006706),GO_Biological_Process_2025,0.275992,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,9,,"HSD17B14,HSD17B6,CYP3A4,HSD17B11,CYP19A1,STS,C..."
5957,Thyroid Hormone Metabolic Process (GO:0042403),GO_Biological_Process_2025,0.200171,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,10,FOXE1,"DIDO1,CPQ,SLC16A2,FOXE1,DIO2,DIO3,SLC5A5,DUOX1..."


In [10]:
scores = "..//..//..//data/for_gsea/geneset_disease_zscore.tsv"
library = "..//..//..//data/gene_sets/gene_sets.gmt"
output_name = "..//..//..//data/gsea_results/geneset_disease_gsea.tsv"

run_gsea_pandas(scores, library, processes=4, output_tsv=output_name)


GSEA input shape: (8285, 2)
GSEA input columns: [1, 0]
GSEA input sample:
        1          0
0  CDKN2B  21.634413
1     FTO  18.318823
2    APOE  15.455358
3     ABO  15.153941
4   SH2B3  12.441185
GSEA results saved to ../../../data/gsea_results/geneset_disease_gsea.tsv


,Term,Source,es,nes,pval,sidak,fdr,geneset_size,leading_edge,propagated_edge
0,Regulation of DNA-templated Transcription (GO:...,GO_Biological_Process_2025,0.543573,6.949171,3.674401e-12,2.189575e-08,2.189575e-08,1058,"INS,APOE,BMP4,SMAD3,TCF7L2,PHF2,SLC39A8,IRF4,C...","HEXIM2,ESX1,PASD1,IL31RA,NOBOX,SKAP1,PTF1A,ATF..."
1,Regulation of Transcription by RNA Polymerase ...,GO_Biological_Process_2025,0.532714,6.621944,3.545044e-11,2.112492e-07,1.056246e-07,1136,"ISL1,BMP4,SMAD3,TCF7L2,AHI1,TNFSF11,IRF4,PHF2,...","HEXIM2,HEXIM1,ESX1,PASD1,ISL1,ISL2,LARP7,TFR2,..."
2,Positive Regulation of DNA-templated Transcrip...,GO_Biological_Process_2025,0.592710,6.411034,1.445356e-10,8.612871e-07,2.870958e-07,770,"APOE,ISL1,TERT,SMAD3,TCF7L2,BMP4,AHI1,IRF4,CHE...","ZNF281,DEK,GATAD2A,RFX3,MRTFB,RFX4,MRTFA,ISL1,..."
3,Regulation of Long-Chain Fatty Acid Import Acr...,GO_Biological_Process_2025,-0.795440,-6.058297,1.375702e-09,8.197773e-06,2.049452e-06,5,"AKT2,AKT1,ACSL5","IRS2,AKT1,AKT2,ACSL5,THBS1"
4,Signal Transduction,Reactome_Pathways_2024,0.476955,5.975668,2.291499e-09,1.365495e-05,2.557062e-06,1412,"CDKN2B,APOE,SH2B3,TERT,SMAD3,TCF7L2,ESR1,MYC,I...","FNBP1,BAD,ANKFY1,FRS3,FRS2,PRKAB1,PRKAB2,SCD,S..."
...,...,...,...,...,...,...,...,...,...,...
5954,Amplification of Signal From Unattached Kineto...,Reactome_Pathways_2024,0.300340,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,42,"SPC24,CENPC,NUF2,SPDL1,B9D2,MAD1L1,KIF18A","CENPU,NUDC,SEC13,KNL1,PPP2R5C,PPP2R5B,PPP2R1A,..."
5955,Amino Acid Transport Across the Plasma Membrane,Reactome_Pathways_2024,0.309508,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,17,,"SLC36A2,SLC36A1,SLC36A4,SLC38A2,SLC38A1,SLC6A1..."
5956,Aerobic Respiration and Respiratory Electron T...,Reactome_Pathways_2024,0.175218,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,92,"UQCC1,SLC25A13","PDHX,TRAP1,ADHFE1,ARMC8,GSTZ1,PC,SCO2,SCO1,TME..."
5957,Endomembrane System Organization (GO:0010256),GO_Biological_Process_2025,0.186000,-0.000000,1.000000e+00,1.000000e+00,1.000000e+00,93,"MYO18A,BCL11B,SPTBN1,SYNE1,MIA3,COG6,RAB2A,BAG...","DYNC2H1,TMEM170A,PLEKHA3,GOLPH3L,MAPK15,DES,NU..."
